# Edit spline in XR

Example of rendering splines and editing them live in XR.

<video src="../figures/spline-editor.webm" controls>

## Setup runner & utilities

The usual setup:

In [1]:
from nanover.app import OmniRunner
from nanover.openmm import OpenMMSimulation

simulation = OpenMMSimulation.from_xml_path("../systems/openmm/17-ala.openmm.zip")

OmniRunner.close_all_runners()
imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="EXAMPLE: spline editor")
imd_runner.print_basic_info()
imd_runner.load(0)

Serving "EXAMPLE: spline editor" (ws://localhost:64902), discoverable on all interfaces on port 54545
Available simulations:
[0]: "17-ala.openmm"
Switched to [0]: "17-ala.openmm"
Switched to [0]: "17-ala.openmm"


In [2]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)

## Spline drawings and interaction

In [3]:
import math
from nanover.utilities.intersection import closest_point_on_polyline
import numpy as np
from nanover.jupyter import Mode


OBJECT_RADIUS = .2
OBJECT_COLOR = [1.0, 0.0, 0.0, 1.0]
HOVER_COLOR = [1.0, 1.0, 1.0, 0.5]
CURSOR_RADIUS = 0.25

# mapping of cursor to currently grabbed object if any
CURSOR_GRABBED_OBJECT_ID = {}
# mapping of object id to object position
OBJECT_POSITIONS = {}
SPLINE_CONTROL_POINT_IDS = []
SPLINE_RASTER_POINTS = []
NEXT_HANDLE_ID = 0


def make_handle(position):
    global NEXT_HANDLE_ID
    handle_id = str(NEXT_HANDLE_ID)
    NEXT_HANDLE_ID += 1
    OBJECT_POSITIONS[handle_id] = position
    return handle_id


def spline_4p(t, p_1, p0, p1, p2):
    # https://stackoverflow.com/questions/1251438/catmull-rom-splines-in-python
    # wikipedia Catmull-Rom -> Cubic_Hermite_spline
    # 0 -> p0,  1 -> p1,  1/2 -> (- p_1 + 9 p0 + 9 p1 - p2) / 16
    # assert 0 <= t <= 1
    return (
          t*((2-t)*t - 1)     * p_1
        + (t*t*(3*t - 5) + 2) * p0
        + t*((4 - 3*t)*t + 1) * p1
        + (t-1)*t*t           * p2 ) / 2


SPLINE_DETAIL = 8

def redraw_spline():
    P = [np.array(OBJECT_POSITIONS[handle_id]) for handle_id in SPLINE_CONTROL_POINT_IDS]
    P = [P[0], *P, P[-1]]  # padding
    SPLINE_RASTER_POINTS.clear()

    count = SPLINE_DETAIL

    for j in range(1, len(P)-2): # skip the ends
        for t in range(count):  # t: 0 .1 .2 .. .9
            p = spline_4p(t/count, P[j-1], P[j], P[j+1], P[j+2])
            SPLINE_RASTER_POINTS.append(p)
    SPLINE_RASTER_POINTS.append(P[-1])

    utilities.objects.update_line("spline", positions=SPLINE_RASTER_POINTS, color=OBJECT_COLOR, size=0.05)


def redraw_object(object_id):
    utilities.objects.update_shape(object_id, position=OBJECT_POSITIONS[object_id], color=OBJECT_COLOR, size=OBJECT_RADIUS)


def intersect_objects(position):
    for object_id, object_pos in OBJECT_POSITIONS.items():
        if np.linalg.norm(np.subtract(position, object_pos)) < OBJECT_RADIUS * .5 + CURSOR_RADIUS:
            return object_id
    return None


def intersect_spline(position, radius):
    result = closest_point_on_polyline(SPLINE_RASTER_POINTS, position)
    return result.index if result.distance <= radius else None


# put some objects in the scene
for i in range(8):
    angle = math.pi * 2 * i / 8
    handle_id = make_handle([5+math.cos(angle) * 3, 5+math.sin(angle) * 3, 5])
    redraw_object(handle_id)
    SPLINE_CONTROL_POINT_IDS.append(handle_id)
redraw_spline()


class MoveObjectMode(Mode):
    def on_button_pressed(self, *, key: str, cursor: dict, button: str):
        next_pos = utilities.scene_transform.point_parent_to_local(cursor["position"])
        hovered = intersect_objects(next_pos)
        available = hovered is not None and hovered not in CURSOR_GRABBED_OBJECT_ID.values()

        # grab hovered object if not already grabbed
        if button == "primary" and available:
            CURSOR_GRABBED_OBJECT_ID[key] = hovered
        # add new control points if none hovered
        elif button == "primary" and not hovered:
            closest = closest_point_on_polyline(SPLINE_RASTER_POINTS, next_pos)
            if closest.distance <= CURSOR_RADIUS:
                index = closest.index // SPLINE_DETAIL
                handle_id = make_handle(next_pos)
                SPLINE_CONTROL_POINT_IDS.insert(index + 1, handle_id)
                CURSOR_GRABBED_OBJECT_ID[key] = handle_id
                redraw_spline()
        # remove control point if available
        elif button == "secondary" and available and len(OBJECT_POSITIONS) > 3:
            OBJECT_POSITIONS.pop(hovered)
            SPLINE_CONTROL_POINT_IDS.remove(hovered)
            utilities.objects.remove_shape(hovered)
            redraw_spline()

    def on_button_released(self, *, key: str, cursor: dict, button: str):
        # release grabbed
        if button == "primary":
            CURSOR_GRABBED_OBJECT_ID.pop(key, None)

    def on_cursor_updated(self, *, key: str, cursor: dict):
        next_pos = utilities.scene_transform.point_parent_to_local(cursor["position"])

        # if this cursor has grabbed something, update its position
        if key in CURSOR_GRABBED_OBJECT_ID:
            object_id = CURSOR_GRABBED_OBJECT_ID[key]
            OBJECT_POSITIONS[object_id] = next_pos
            redraw_object(object_id)
            redraw_spline()

        # show/remove hover graphic is this cursor hovers something
        hovered = intersect_objects(next_pos)
        if hovered is None:
            closest = closest_point_on_polyline(SPLINE_RASTER_POINTS, next_pos)
            if closest.distance <= CURSOR_RADIUS:
                utilities.objects.update_shape(f"hovered.{key}", position=closest.point, color=HOVER_COLOR, size=OBJECT_RADIUS * .9)
            else:
                utilities.objects.remove_shape(f"hovered.{key}")
        else:
            utilities.objects.update_shape(f"hovered.{key}", position=OBJECT_POSITIONS[hovered], color=HOVER_COLOR, size=OBJECT_RADIUS * 1.1)


utilities.add_interaction_mode(MoveObjectMode(), "edit spline", icon="✊")
utilities.modes.enter_mode("edit spline")